In [33]:
# Install streamlit dan pyngrok
!pip install streamlit pyngrok --quiet


In [34]:
%%writefile apps.py
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

TARGET_COL = "Klasifikasi Kemiskinan"

@st.cache_data
def load_data():
    # Pastikan file CSV ini sudah ada di folder kerja / Colab
    df = pd.read_csv("Klasifikasi Tingkat Kemiskinan di Indonesia.csv", sep=";")

    # Bersihkan nama kolom
    df.columns = df.columns.str.strip()

    # Buang kolom teks yang tidak dipakai model
    df = df.drop(columns=["Provinsi", "Kab/Kota"], errors="ignore")

    # Ubah semua kolom selain target menjadi numerik
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.replace(",", ".", regex=False)
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Hapus data kosong
    df = df.dropna()

    # Target dibuat integer
    df[TARGET_COL] = df[TARGET_COL].astype(int)

    return df

@st.cache_data
def train_model(df):
    X = df.drop(TARGET_COL, axis=1)
    y = df[TARGET_COL]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    accuracy = model.score(X_test, y_test)

    return model, accuracy, X.columns

def main():
    st.set_page_config(page_title="Klasifikasi Kemiskinan", layout="centered")
    st.title("📊 Prediksi Tingkat Kemiskinan Indonesia")
    st.write("Masukkan nilai indikator pada wilayah yang ingin diprediksi.")

    df = load_data()
    model, accuracy, features = train_model(df)

    st.info(f"Jumlah data bersih: {len(df)} baris")
    st.info(f"Akurasi model pada data uji: {accuracy*100:.2f}%")

    with st.form("form_prediksi"):
        input_data = []

        for col in features:
            min_val = float(df[col].min())
            max_val = float(df[col].max())
            mean_val = float(df[col].mean())

            val = st.number_input(
                label=col,
                min_value=min_val,
                max_value=max_val,
                value=mean_val,
                step=(max_val - min_val) / 100 if max_val > min_val else 1.0
            )
            input_data.append(val)

        submitted = st.form_submit_button("Prediksi")

    if submitted:
        input_array = np.array([input_data])
        prediction = model.predict(input_array)[0]

        hasil = "Tinggi" if int(prediction) == 1 else "Rendah"
        st.success(f"Hasil Prediksi Kemiskinan: **{hasil}**")
        st.write(f"Label asli model: **{int(prediction)}**")

    if st.checkbox("Tampilkan Dataset"):
        st.dataframe(df)

if __name__ == "__main__":
    main()

Overwriting apps.py


In [35]:
# Jalankan streamlit dengan expose port pakai ngrok
from pyngrok import ngrok
ngrok.set_auth_token("3DF10Gxm1EU6JHhdULznbi8jqg3_5h2t2ETKpJLFU2BwS99CK")
public_url = ngrok.connect(8501)
print(f"Streamlit app bisa diakses di:\n{public_url}")

!streamlit run apps.py &>/dev/null&

Streamlit app bisa diakses di:
NgrokTunnel: "https://ferocity-hangnail-stilt.ngrok-free.dev" -> "http://localhost:8501"
